### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [ ]:
from src.utility import process_all_pdfs

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

In [ ]:
all_pdf_documents

In [ ]:
from src.utility import split_documents

chunks=split_documents(all_pdf_documents)
chunks

### embedding And vectorStoreDB

In [ ]:
from src.embedding_manager import EmbeddingManager

## initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager


### VectorStore

In [ ]:
from src.vector_store import VectorStore

vectorstore=VectorStore()
vectorstore
    

In [ ]:
chunks

In [ ]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

### Retriever Pipeline From VectorStore

In [ ]:
from src.rag_retriever import RAGRetriever

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [ ]:
rag_retriever.retrieve("What is attention is all you need")

In [ ]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

### RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain.messages import HumanMessage, SystemMessage

In [ ]:
from src.grok_llm import GroqLLM
from data.constants import groq_api_key

# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(groq_api_key)
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

In [ ]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Vector semantics is the standard way")

In [ ]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

### Integration Vectordb Context pipeline With LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from src.rag_pipeline import RAGPipeline
from data.constants import groq_api_key
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

adv_rag = RAGPipeline(rag_retriever, llm)

In [ ]:
answer=adv_rag.query_simple("what is attention mechanism")
print(answer)

### Enhanced RAG Pipeline Features

In [ ]:

# Example usage:
result = adv_rag.query_advanced("Hard Negative Mining Technqiues", top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
result = adv_rag.query_formatted("what is attention mechanism", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

In [ ]:
result = adv_rag.query_formatted("Unsupervised and weakly supervised learnin", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])